<a href="https://colab.research.google.com/github/mwasjack/slides/blob/ccr-ffe47fa4-vrazz8/casselle_fb_render.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

thecust_casselle_fb_data_path = kagglehub.dataset_download('thecust/casselle-fb-data')

print('Data source import complete.')


In [ ]:
# CASSELLE Facebook ad: every 3D shot in Cycles on the Kaggle GPU. Same Blender scripts as the laptop.
import glob, json, os, shutil, subprocess, time, zipfile
W, TMP, VER = "/kaggle/working", "/kaggle/temp", "4.2.11"
os.makedirs(TMP, exist_ok=True)


def run(c):
    print("$", c[:300], flush=True)
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:], flush=True)
    return r


t0 = time.time()
# the data: one zip (or Kaggle's unpacked copy of it) -> a writable tree
ROOT = f"{TMP}/root"
hits = glob.glob("/kaggle/input/**/casselle_fb_shots.py", recursive=True)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), ROOT)
else:
    z = glob.glob("/kaggle/input/**/casselle-fb.zip", recursive=True)[0]
    zipfile.ZipFile(z).extractall(TMP)
    os.rename(f"{TMP}/casselle-fb", ROOT)
print("ROOT", os.listdir(ROOT), flush=True)
run("apt-get -qq update && apt-get -qq install -y libxrender1 libxi6 libxkbcommon0 libsm6 libxxf86vm1 libxfixes3 libgl1 libegl1 > /dev/null 2>&1")
run(f"cd {TMP} && wget -q https://download.blender.org/release/Blender4.2/blender-{VER}-linux-x64.tar.xz -O b.tar.xz && tar -xf b.tar.xz && rm b.tar.xz")
run("pip -q install imageio-ffmpeg")
import imageio_ffmpeg  # noqa: E402
FF = imageio_ffmpeg.get_ffmpeg_exe()
BL = f"{TMP}/blender-{VER}-linux-x64/blender"
for d in ("clips", "anchors", "keys"):
    os.makedirs(f"{W}/{d}", exist_ok=True)
SHOTS = [("hero", 64), ("colourway", 64), ("match", 64), ("cuff", 48), ("belt", 48), ("hem", 48), ("endcard", 64),
         ("petals", 48)]
report = {}
for shot, spp in SHOTS:
    t = time.time()
    out = f"{TMP}/frames/{shot}/"
    if shot == "hero":
        cmd = f"{BL} -b --factory-startup -P {ROOT}/blender/casselle_hero.py -- --out {out} --engine cycles --samples {spp}"
    else:
        cmd = (f"{BL} -b --factory-startup -P {ROOT}/blender/casselle_fb_shots.py -- --shot {shot} --out {out} "
               f"--engine cycles --samples {spp}")
    r = run(cmd)
    fr = sorted(f for f in os.listdir(out) if f.startswith("frame_")) if os.path.isdir(out) else []
    if fr:
        if shot == "petals":  # keeps its alpha: ship the PNG frames
            shutil.make_archive(f"{W}/clips/petals-frames", "zip", out)
        else:
            run(f"{FF} -v error -y -framerate 30 -i {out}frame_%04d.png -c:v libx264 -preset slow -crf 10 "
                f"-pix_fmt yuv420p -movflags +faststart {W}/clips/{shot}.mp4")
        for k in (fr[0], fr[len(fr) // 2], fr[-1]):
            shutil.copy(out + k, f"{W}/keys/{shot}-{k}")
    if os.path.exists(out + "anchors.json"):
        shutil.copy(out + "anchors.json", f"{W}/anchors/{shot}.json")
    shutil.rmtree(out, ignore_errors=True)
    report[shot] = {"seconds": round(time.time() - t, 1), "frames": len(fr), "rc": r.returncode,
                    "gpu": [l for l in (r.stdout + r.stderr).splitlines() if "GPU" in l or "OptiX" in l][:3]}
    report["elapsed"] = round(time.time() - t0, 1)
    json.dump(report, open(f"{W}/report.json", "w"), indent=1)
    print(report, flush=True)
